<a href="https://colab.research.google.com/github/tmvenkadesh/AgenticAI/blob/main/Agentic_demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!curl -LsSf https://astral.sh/uv/install.sh | sh


downloading uv 0.10.9 x86_64-unknown-linux-gnu
no checksums to verify
installing to /usr/local/bin
  uv
  uvx
everything's installed!


In [2]:
!uv pip install langchain-ollama langgraph langfuse pydantic python-dotenv --system


Using Python 3.12.12 environment at: /usr
Resolved 52 packages in 962ms
Prepared 6 packages in 171ms
Uninstalled 2 packages in 7ms
Installed 6 packages in 46ms
 + backoff==2.2.1
 + langchain-ollama==1.0.1
 + langfuse==4.0.0
 + ollama==0.6.1
 - packaging==26.0
 + packaging==25.0
 - wrapt==2.1.1
 + wrapt==1.17.3


In [3]:
!apt-get install -y zstd


Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  zstd
0 upgraded, 1 newly installed, 0 to remove and 2 not upgraded.
Need to get 603 kB of archives.
After this operation, 1,695 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/main amd64 zstd amd64 1.4.8+dfsg-3build1 [603 kB]
Fetched 603 kB in 1s (606 kB/s)
Selecting previously unselected package zstd.
(Reading database ... 117540 files and directories currently installed.)
Preparing to unpack .../zstd_1.4.8+dfsg-3build1_amd64.deb ...
Unpacking zstd (1.4.8+dfsg-3build1) ...
Setting up zstd (1.4.8+dfsg-3build1) ...
Processing triggers for man-db (2.10.2-1) ...


In [5]:
!curl -fsSL https://ollama.com/install.sh | sh


>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [6]:
import subprocess
subprocess.Popen(["ollama", "serve"])
import time
time.sleep(3)  # wait for server to start


In [7]:
!ollama pull llama3.2


In [8]:
from google.colab import userdata
import os
os.environ["LANGFUSE_PUBLIC_KEY"] =userdata.get("LANGFUSE_PUBLIC_KEY")
os.environ["LANGFUSE_SECRET_KEY"]=userdata.get("LANGFUSE_SECRET_KEY")
os.environ["LANGFUSE_BASE_URL"]="https://cloud.langfuse.com"

In [9]:
from langchain_ollama import ChatOllama
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from langgraph.graph import StateGraph, END
from langgraph.prebuilt import ToolNode
from typing import Annotated, TypedDict
from langfuse.langchain import CallbackHandler
from langfuse import get_client
from dotenv import load_dotenv

load_dotenv()

langfuse = get_client()
langfuse_handler = CallbackHandler()

config = {
    "callbacks": [langfuse_handler]
}


# --- Tools ---

@tool
def handle_technical_query(question: str) -> str:
    """Handle technical, programming, coding, software engineering,
    or computer science related questions. Use this tool when the user
    asks about writing code, scripts, debugging, algorithms, frameworks,
    APIs, databases, JSON parsing, or any technology-related topic."""
    print("--Handling Technical query via tool call--")
    return f"Here is some python code for: {question}"


@tool
def handle_general_query(question: str) -> str:
    """Handle general knowledge questions like geography, history, science,
    culture, or everyday topics. Use this tool ONLY when the question is NOT
    about programming or technology."""
    print("--Handling General query via tool call--")
    return f"Here is your general question's answer for: {question}"


tools = [handle_technical_query, handle_general_query]

# --- LLM with tools bound ---
llm = ChatOllama(model="llama3.2", temperature=0)
llm_with_tools = llm.bind_tools(tools)


# --- State ---
class AgentState(TypedDict):
    messages: Annotated[list, "The conversation messages"]


# ===========================================================================
# HUMAN VALIDATION OF LLM OUTPUT
# ===========================================================================
# The LLM returns tool_calls like:
#   {"name": "handle_technical_query", "args": {"question": "..."}}
#
# But Llama 3.2 sometimes sends BROKEN args like:
#   {"name": "handle_technical_query", "args": {"question": {"type": "string"}}}
#
# We validate and fix this BEFORE ToolNode sees it.
# ===========================================================================

# Rule 1: What tool names are allowed?
VALID_TOOL_NAMES = {"handle_technical_query", "handle_general_query"}

# Rule 2: What args does each tool expect?
EXPECTED_ARGS = {
    "handle_technical_query": {"question": str},
    "handle_general_query": {"question": str},
}


def validate_tool_calls(tool_calls: list, user_question: str) -> list:
    """Validate and fix LLM tool call output using human-written rules.

    Checks:
    1. Is the tool name valid?
    2. Are all required args present?
    3. Is each arg the correct type?
    4. If an arg is wrong type (dict instead of str), extract or fallback.
    """
    validated = []

    for tc in tool_calls:
        tool_name = tc["name"]
        args = tc["args"]

        # CHECK 1: Is the tool name valid?
        if tool_name not in VALID_TOOL_NAMES:
            print(f"[VALIDATION] Unknown tool '{tool_name}', skipping.")
            continue

        expected = EXPECTED_ARGS[tool_name]
        fixed_args = {}

        for arg_name, expected_type in expected.items():

            # CHECK 2: Is the required arg present?
            if arg_name not in args:
                print(f"[VALIDATION] Missing arg '{arg_name}', using user question.")
                fixed_args[arg_name] = user_question
                continue

            value = args[arg_name]

            # CHECK 3: Is the arg the correct type?
            if isinstance(value, expected_type):
                fixed_args[arg_name] = value
                continue

            # CHECK 4: Arg is wrong type — try to extract the real value
            if isinstance(value, dict):
                # Llama 3.2 sends: {"type": "string", "value": "actual text"}
                # or sometimes just: {"type": "string"} with no value at all
                extracted = value.get("value") or value.get("content") or value.get("description")

                if extracted and isinstance(extracted, expected_type):
                    print(f"[VALIDATION] Fixed '{arg_name}': extracted from dict → '{extracted}'")
                    fixed_args[arg_name] = extracted
                else:
                    print(f"[VALIDATION] Can't extract '{arg_name}' from {value}, using user question.")
                    fixed_args[arg_name] = user_question
            else:
                # Some other unexpected type — convert to string
                print(f"[VALIDATION] Unexpected type for '{arg_name}': {type(value)}, converting.")
                fixed_args[arg_name] = str(value)

        tc["args"] = fixed_args
        validated.append(tc)

    return validated


# --- Graph Nodes ---

def call_llm(state: AgentState):
    """LLM decides which tool to call based on the user's question."""
    messages = state["messages"]
    response = llm_with_tools.invoke(messages)

    print("llm response :", response)

    # HUMAN VALIDATION: clean up LLM output before ToolNode processes it
    if response.tool_calls:
        user_question = messages[-1].content
        response.tool_calls = validate_tool_calls(response.tool_calls, user_question)

    return {"messages": messages + [response]}


def route_after_llm(state: AgentState):
    """Check if the LLM made a tool call or responded directly."""
    last_message = state["messages"][-1]
    if last_message.tool_calls:
        return "tools"
    return END


# ToolNode automatically executes whatever tool the LLM chose
tool_node = ToolNode(tools)

# --- Build Graph ---
workflow = StateGraph(AgentState)

workflow.add_node("llm", call_llm)
workflow.add_node("tools", tool_node)

workflow.set_entry_point("llm")

workflow.add_conditional_edges(
    "llm",
    route_after_llm,
    {
        "tools": "tools",
        END: END
    }
)

workflow.add_edge("tools", END)

app = workflow.compile()


# --- Testing ---

print("=== Technical Question ===")
result = app.invoke(
    {"messages": [HumanMessage(content="write some python script to parse json")]},
    config=config
)
print("Tool used:", result["messages"][-1].content)

print()

print("=== General Question ===")
result = app.invoke(
    {"messages": [HumanMessage(content="capital of india")]},
    config=config
)
print("Tool used:", result["messages"][-1].content)


=== Technical Question ===
llm response : content='' additional_kwargs={} response_metadata={'model': 'llama3.2', 'created_at': '2026-03-11T14:30:04.363329137Z', 'done': True, 'done_reason': 'stop', 'total_duration': 64039358833, 'load_duration': 3473016928, 'prompt_eval_count': 261, 'prompt_eval_duration': 52242692978, 'eval_count': 24, 'eval_duration': 8304354576, 'logprobs': None, 'model_name': 'llama3.2', 'model_provider': 'ollama'} id='lc_run--019cdd4d-0d0d-7763-bd92-657581dbf949-0' tool_calls=[{'name': 'handle_technical_query', 'args': {'question': {'type': 'string'}}, 'id': '8fb6d54b-11d1-4af2-a061-6334d2c18ccc', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input_tokens': 261, 'output_tokens': 24, 'total_tokens': 285}
[VALIDATION] Can't extract 'question' from {'type': 'string'}, using user question.
--Handling Technical query via tool call--
Tool used: Here is some python code for: write some python script to parse json

=== General Question ===
llm response : c